# שיעור 27 — רגרסיה ליניארית: פתרון מתועד
135 דקות מוצעות (שלוש שעות לימוד). ידע קודם: X/y, חלוקה, השלמה מאימון, fit/predict, ממוצע ותרשים פיזור.

**פתיחה:** דודו שואל אם אורך סנפיר יכול לעזור להעריך מסה של פינגווין שלא נשקל. הפלט הוא מספר בגרמים, לא שם מין. זו דוגמת מחקר לימודית; תחזית אינה תחליף אוטומטי למדידה.
באותו מאגר אפשר להגדיר שאלות שונות: סיווג species לעומת רגרסיה של body_mass_g. מספר שמשמש קוד קטגוריה אינו הופך יעד לרגרסיה — המשמעות קובעת.

עוגן: [סיכום הרגרסיה ביחידה 3](../../04-Introduction-to-ML/Linear-Regression-Intro/Didactic-Summary.md) ו[מחברת המבוא](../../04-Introduction-to-ML/Linear-Regression-Intro/Linear-Regression-Intro.ipynb). זוהי התאמה שלנו מ־Iris ל־Penguins והוספת הערכה ב־Test. חומרי המקור נשמרו; אין כאן תרגיל מקור חדש מהמרצה.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

In [ ]:
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df[["flipper_length_mm", "bill_length_mm", "body_mass_g"]].isna().sum())

## 1. יעד מספרי והכנה
הרשומה היא פינגווין בעל מסה ידועה. X: אורך מקור במ״מ, y: מסה בגרמים. לא מכניסים את המסה לקלט ולא משלימים יעד חסר. ההשלמה ל־X נלמדת מאימון; נתון זה אינו משתמש בתוצאות הבדיקה.

In [ ]:
# לא משלימים יעד חסר; משתמשים בעותק עבודה
model_df = df.dropna(subset=["body_mass_g"]).copy()
feature = "bill_length_mm"
X = model_df[[feature]].copy()
y = model_df["body_mass_g"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17
)
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)
print("Eligible:", len(model_df), "Train:", len(X_train), "Test:", len(X_test))
print("Train feature range:", X_train_ready[feature].min(), X_train_ready[feature].max())

## 2. קו ותחזית
משמעות השיפוע היא שינוי בתחזית המסה בגרמים לכל מ״מ נוסף באורך המקור, במסגרת המודל ובטווח הנתונים. אין כאן הוכחה לסיבתיות. החיתוך ב־x=0 עשוי להיות חסר משמעות שימושית.

In [ ]:
model = LinearRegression()
model.fit(X_train_ready, y_train)
predictions = model.predict(X_test_ready)
w = model.coef_[0]
b = model.intercept_
print("Slope (g/mm):", w, "Intercept (g):", b)
new_penguin = pd.DataFrame({feature: [45]})
new_prediction = model.predict(new_penguin)
print("New feature value (mm):", 45, "Predicted mass (g):", new_prediction[0])

In [ ]:
print("45 mm within training range:",
      X_train_ready[feature].min() <= 45 <= X_train_ready[feature].max())

## 3. תרשים ושאריות
הקו מותאם לאימון בלבד. actual−predicted חיובי פירושו הערכת חסר; שלילי הערכת יתר. קוראים דוגמאות אמיתיות מהטבלה, ולא ממציאים שגיאות.

In [ ]:
# קטע ציור נתון; ממיינים את X כדי לצייר ישר בלי חיבור בזיגזג
line_X = X_train_ready.sort_values(feature)
line_y = model.predict(line_X)
plt.figure()
plt.scatter(X_train_ready[feature], y_train, alpha=0.35, label="Train")
plt.scatter(X_test_ready[feature], y_test, alpha=0.65, label="Test")
plt.plot(line_X[feature], line_y, color="black", label="Line fitted on Train")
plt.xlabel(feature + " (mm)")
plt.ylabel("body_mass_g (g)")
plt.title("Penguins — one-feature linear regression")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame(
    {"actual_g": y_test, "predicted_g": predictions}, index=y_test.index
)
comparison["residual_g"] = comparison["actual_g"] - comparison["predicted_g"]
display(comparison.head(8))
plt.figure()
plt.scatter(comparison["predicted_g"], comparison["residual_g"], alpha=0.6)
plt.axhline(0, color="black")
plt.xlabel("Predicted mass (g)")
plt.ylabel("Residual: actual - predicted (g)")
plt.title("Test residuals")
plt.tight_layout()
plt.show()

## 4. הערכה
מחשבים את הפיזור סביב ממוצע הבדיקה לצורך הגדרת R² אחרי החיזוי. ה־baseline לומד ממוצע אימון בלבד. ממוצע האימון אינו בהכרח ממוצע הבדיקה ולכן R² שלו אינו בהכרח אפס.

In [ ]:
# ממוצע y_test משמש רק להגדרת המדד, לא ליצירת תחזיות
sse = (comparison["residual_g"] ** 2).sum()
test_mean = y_test.mean()
sst = ((y_test - test_mean) ** 2).sum()
r2_manual = 1 - sse / sst
r2_test = r2_score(y_test, predictions)
print("SSE:", sse, "SST:", sst)
print("R² manual:", r2_manual, "R² library:", r2_test)

# קו בסיס מעשי: חיזוי ממוצע שנלמד מאימון בלבד
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_ready, y_train)
baseline_predictions = baseline.predict(X_test_ready)
baseline_r2 = r2_score(y_test, baseline_predictions)
print("Training mean baseline (g):", baseline.constant_[0, 0])
print("Baseline R² on same Test:", baseline_r2)

## 5. כרטיס מודל והסבר
המקור והיחידות כפי שתועדו; 342 בעלי יעד ידוע חולקו יחד ל־75% אימון ו־25% בדיקה, seed=17. המודל מתאים קו יחיד מאורך מקור למסה; מקדמים, תחזית 45 מ״מ, שאריות וציוני Test נלקחים מפלט ההרצה.
מגבלות: מדידה אחת אינה מתארת מין, גיל, מין ביולוגי וגורמים אחרים; דגימה ממקום/תקופה אחרים עלולה לשנות את הקשר. אין להניח שהישר מתאר קשר סיבתי או שתחזית חדשה מדויקת.
R² אינו אחוז תחזיות נכונות. ערך שלילי אפשרי כאשר SSE>SST; הדוגמה [10,20,30] מול [40,40,40] נותנת −6. לא בודקים שוויון מדויק בין תחזית מספרית למדידה כדי להגדיר Accuracy.
גם אם הציון שונה מן המודרכת, אין להסיק על עדיפות מאפיין: החלוקה שונה. לא מחפשים מאפיין חדש בעזרת Test ומדווחים אותה בדיקה כאילו נותרה עצמאית. בהמשך נלמד אימות בתוך אימון.

### תוצאות ההרצה שנבדקה בעת הכנת הערכה
256 דוגמאות אימון ו־86 בדיקה. השיפוע כ־91.7301 גרם/מ״מ והחיתוך כ־144.436 גרם. תחזית עבור מקור 45 מ״מ: כ־4272.29 גרם. R² בבדיקה כ־0.12433, מול כ־−0.07512 לקו בסיס ממוצע אימון. זו התאמה מוגבלת, ולא תקלה שיש לתקן באמצעות חיפוש על Test. יש לקרוא את הפלט מחדש אם הנתונים או הספרייה משתנים. אין להשוות לציון המודרכת כהוכחת עדיפות של מאפיין, כי החלוקה שונה.